# 📊 W3: 공정 KPI 분석 & 간트차트
**hexa-5 — Week 3** | ⏱️ 60분 | 🎯 공정지연 자동감지 + 간트차트 + Gemini

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, pandas as pd
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 공정 정보 입력 (✏️)

In [ ]:
INFO={'name':'✏️ [교육 기업명/건설사]','site':'✏️ [현장명]'}
import pandas as pd
PROCESS=pd.DataFrame({
    '공정':['기초공사','골조공사','설비공사','마감공사','준공검사'],
    '계획시작':['2026-01-01','2026-02-01','2026-03-01','2026-04-01','2026-05-01'],
    '계획종료':['2026-01-31','2026-02-28','2026-03-31','2026-04-30','2026-05-15'],
    '실제시작':['2026-01-01','2026-02-05','2026-03-10','2026-04-01',''],
    '진행률':[100,100,70,20,0]
})
print(PROCESS.to_string(index=False))


## Step 2: 지연 공정 자동 감지

In [ ]:
PROCESS['지연여부']=[True if s else False for s in PROCESS['실제시작']]
delayed=PROCESS[PROCESS['진행률']<50]
print(f'⚠️ 지연/저조 공정: {delayed["공정"].tolist()}')


## Step 3: 간트차트 생성

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
fig,ax=plt.subplots(figsize=(12,5))
colors={'100':'#2ecc71','70':'#f39c12','20':'#e74c3c','0':'#bdc3c7'}
for i,row in PROCESS.iterrows():
    start=pd.to_datetime(row['계획시작']); end=pd.to_datetime(row['계획종료'])
    dur=(end-start).days
    clr='#2ecc71' if row['진행률']>=80 else '#f39c12' if row['진행률']>=50 else '#e74c3c'
    ax.barh(row['공정'],dur,left=mdates.date2num(start),color=clr,alpha=0.8)
    ax.text(mdates.date2num(start)+dur/2,i,f"{row['진행률']}%",ha='center',va='center',color='white',fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.set_title(f'{INFO["site"]} 공정 간트차트'); ax.set_xlabel('날짜')
plt.tight_layout(); plt.savefig('gantt.png',dpi=150,bbox_inches='tight'); plt.show()


## Step 4: Gemini 공정 분석

In [ ]:
p=f"""건설 공정 관리 전문가로서 아래 현황을 분석하세요.
현장: {INFO['site']}
지연/저조 공정: {delayed['공정'].tolist()}
원인 분석 + 즉시 조치사항 + 공기 단축 방안. 마크다운."""
resp=model.generate_content(p)
print(resp.text)


## Step 5: 다운로드

In [ ]:
PROCESS.to_csv('process_kpi.csv',index=False,encoding='utf-8-sig')
with open('process_report.md','w',encoding='utf-8') as f: f.write(resp.text)
from google.colab import files
files.download('process_kpi.csv'); files.download('gantt.png'); files.download('process_report.md')
print('✅ 완료!')
